In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
LOG_DIR = "../logs/"

AGG_LOG = "he-aggregation-service"
CLIENT1_LOG = "he-client-1-go"
CLIENT2_LOG = "he-client-2-go"

In [3]:
agg_time = pd.read_csv(f"{LOG_DIR}{AGG_LOG}-time.csv")
client1_time = pd.read_csv(f"{LOG_DIR}{CLIENT1_LOG}-time.csv")
client2_time = pd.read_csv(f"{LOG_DIR}{CLIENT2_LOG}-time.csv")

agg_time["Service"] = "AggregationService"
client1_time["Service"] = "Client1"
client2_time["Service"] = "Client2"

client1_time.head()

,Date,EndTime,Name,Runtime,Service
0,2025/07/22,16:42:45.854911,DataSpaceClientService - RegisterClient,17.285958ms,Client1
1,2025/07/22,16:42:45.863316,HEService - SetParameters,8.358917ms,Client1
2,2025/07/22,16:42:50.867832,HEService - PartialShareAggregation,1.084µs,Client1
3,2025/07/22,16:42:50.872998,EncryptionHandler - handleReceivePublicKey,1.315917ms,Client1
4,2025/07/22,16:42:50.904355,HEService - PartialRelinKeyAggregation,2.48075ms,Client1


In [4]:
data_joined = pd.concat(
    [agg_time, client1_time, client2_time],
    axis=0)

data_joined = data_joined.reset_index(drop=True)

data_joined

,Date,EndTime,Name,Runtime,Service
0,2025/07/22,16:42:45.841870,addClient,1.708µs,AggregationService
1,2025/07/22,16:42:51.065956,startEncryptionSetupPhaseFor,5.224059s,AggregationService
2,2025/07/22,16:42:53.846724,addClient,1.292µs,AggregationService
3,2025/07/22,16:42:59.221070,startEncryptionSetupPhaseFor,5.37431925s,AggregationService
4,2025/07/22,16:43:09.286822,requestClientTraining,4.1685ms,AggregationService
5,2025/07/22,16:43:25.141169,aggregateWeights,56.834709ms,AggregationService
6,2025/07/22,16:43:25.538867,initiateKeySwitchGeneration,397.6255ms,AggregationService
7,2025/07/22,16:43:28.692864,updateClientModels,2.229893459s,AggregationService
8,2025/07/22,16:43:32.626307,requestClientTraining,3.368667ms,AggregationService
9,2025/07/22,16:43:50.128681,aggregateWeights,54.808708ms,AggregationService


In [5]:
def parse_duration_to_ms(time_str):
    """Parse milliseconds, microseconds, nanoseconds or seconds from a string."""
    if "ms" in time_str:
        return float(time_str.replace("ms", "").strip())
    elif "us" in time_str:
        return float(time_str.replace("us", "").strip()) / 1000.0
    elif "µs" in time_str:
        return float(time_str.replace("µs", "").strip()) / 1000.0
    elif "ns" in time_str:
        return float(time_str.replace("ns", "").strip()) / 1_000_000.0
    elif "s" in time_str:
        return float(time_str.replace("s", "").strip()) * 1000.0
    else:
        raise ValueError(f"Unknown time format: {time_str}")


def create_timeline_offset(df):
    """Create a timeline with an offset for the x-axis."""
    min_time = df["EndTime"].min()
    df["StartMs"] = (pd.to_datetime(df["EndTime"], format="%H:%M:%S.%f") - pd.to_datetime(min_time, format="%H:%M:%S.%f")).dt.total_seconds() * 1000
    
    return df
    
#agg_time["runtime_ms"] = agg_time["Runtime"].apply(parse_duration_to_ms)
#client1_time["runtime_ms"] = client1_time["Runtime"].apply(parse_duration_to_ms)
#client2_time["runtime_ms"] = client2_time["Runtime"].apply(parse_duration_to_ms)
data_joined["runtime_ms"] = data_joined["Runtime"].apply(parse_duration_to_ms)

#agg_time = create_timeline_offset(agg_time)
#client1_time = create_timeline_offset(client1_time)
#client2_time = create_timeline_offset(client2_time)
data_joined = create_timeline_offset(data_joined)

data_joined["EndMs"] = data_joined["StartMs"] + data_joined["runtime_ms"]

data_joined = data_joined.sort_values(by=["StartMs", "EndMs"])
data_joined.reset_index(drop=True, inplace=True)
data_joined

,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs
0,2025/07/22,16:42:45.841870,addClient,1.708µs,AggregationService,0.001708,0.000,0.001708
1,2025/07/22,16:42:45.854911,DataSpaceClientService - RegisterClient,17.285958ms,Client1,17.285958,13.041,30.326958
2,2025/07/22,16:42:45.863316,HEService - SetParameters,8.358917ms,Client1,8.358917,21.446,29.804917
3,2025/07/22,16:42:50.867832,HEService - PartialShareAggregation,1.084µs,Client1,0.001084,5025.962,5025.963084
4,2025/07/22,16:42:50.872998,EncryptionHandler - handleReceivePublicKey,1.315917ms,Client1,1.315917,5031.128,5032.443917
5,2025/07/22,16:42:50.904355,HEService - PartialRelinKeyAggregation,2.48075ms,Client1,2.480750,5062.485,5064.965750
6,2025/07/22,16:42:51.065956,startEncryptionSetupPhaseFor,5.224059s,AggregationService,5224.059000,5224.086,10448.145000
7,2025/07/22,16:42:53.846724,addClient,1.292µs,AggregationService,0.001292,8004.854,8004.855292
8,2025/07/22,16:42:53.857312,DataSpaceClientService - RegisterClient,12.761208ms,Client2,12.761208,8015.442,8028.203208
9,2025/07/22,16:42:53.865943,HEService - SetParameters,8.578583ms,Client2,8.578583,8024.073,8032.651583


In [6]:
print(pio.templates)

pio.templates["research"] = go.layout.Template(
    layout=dict(
        font=dict(color="#000000"),
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        hovermode="closest",
        xaxis=dict(
            tickangle=-45,
            showline=True,
            linewidth=1,
            linecolor="#373737",
            ticks="inside",
            showgrid=True,
            gridcolor="#373737",
            zeroline=True,
            zerolinecolor="#373737",
            zerolinewidth=1,
            mirror=True
        ),
        yaxis=dict(
            showline=True,
            linewidth=1,
            linecolor="#373737",
            ticks="inside",
            showgrid=True,
            gridcolor="#373737",
            zeroline=True,
            zerolinecolor="#373737",
            zerolinewidth=1,
            mirror=True
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.5,
            xanchor="right",
            x=0.95
        ),
    )
)

pio.templates.default = "plotly_white+research"

Templates configuration
-----------------------
    Default template: 'plotly'
    Available templates:
        ['ggplot2', 'seaborn', 'simple_white', 'plotly',
         'plotly_white', 'plotly_dark', 'presentation', 'xgridoff',
         'ygridoff', 'gridon', 'none']



In [15]:
import plotly.express as px
import plotly.graph_objects as go


df = px.data.tips()
fig = px.bar(data_joined[data_joined["runtime_ms"] >= 100], base="StartMs", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], labels={"runtime_ms": "Runtime in millisecond", "Service": "Service"}, color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)
fig.show()

In [8]:
# remove all waiting times

data_joined.at[data_joined.index[0], "Start_without_offset"] = data_joined.at[data_joined.index[0], "StartMs"]
data_joined.at[data_joined.index[0], "End_without_offset"] = data_joined.at[data_joined.index[0], "EndMs"]

for i in range(1, len(data_joined)):
    data_joined.at[data_joined.index[i], "Start_without_offset"] = data_joined.iloc[i-1]["End_without_offset"]
    data_joined.at[data_joined.index[i], "End_without_offset"] = data_joined.at[data_joined.index[i], "Start_without_offset"] + data_joined.at[data_joined.index[i], "runtime_ms"]
data_joined

,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs,Start_without_offset,End_without_offset
0,2025/07/22,16:42:45.841870,addClient,1.708µs,AggregationService,0.001708,0.000,0.001708,0.000000,0.001708
1,2025/07/22,16:42:45.854911,DataSpaceClientService - RegisterClient,17.285958ms,Client1,17.285958,13.041,30.326958,0.001708,17.287666
2,2025/07/22,16:42:45.863316,HEService - SetParameters,8.358917ms,Client1,8.358917,21.446,29.804917,17.287666,25.646583
3,2025/07/22,16:42:50.867832,HEService - PartialShareAggregation,1.084µs,Client1,0.001084,5025.962,5025.963084,25.646583,25.647667
4,2025/07/22,16:42:50.872998,EncryptionHandler - handleReceivePublicKey,1.315917ms,Client1,1.315917,5031.128,5032.443917,25.647667,26.963584
5,2025/07/22,16:42:50.904355,HEService - PartialRelinKeyAggregation,2.48075ms,Client1,2.480750,5062.485,5064.965750,26.963584,29.444334
6,2025/07/22,16:42:51.065956,startEncryptionSetupPhaseFor,5.224059s,AggregationService,5224.059000,5224.086,10448.145000,29.444334,5253.503334
7,2025/07/22,16:42:53.846724,addClient,1.292µs,AggregationService,0.001292,8004.854,8004.855292,5253.503334,5253.504626
8,2025/07/22,16:42:53.857312,DataSpaceClientService - RegisterClient,12.761208ms,Client2,12.761208,8015.442,8028.203208,5253.504626,5266.265834
9,2025/07/22,16:42:53.865943,HEService - SetParameters,8.578583ms,Client2,8.578583,8024.073,8032.651583,5266.265834,5274.844417


In [ ]:
df = px.data.tips()
fig = px.bar(data_joined, base="Start_without_offset", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], labels={"runtime_ms": "Runtime in millisecond", "Service": "Service"}, color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)
fig.show()

In [10]:
data_joined.reset_index(drop=True, inplace=True)

first_occ_encrypt = data_joined.loc[data_joined["Name"] == "requestClientTraining"].index[0]
print(first_occ_encrypt)

initializiation_data = data_joined[:first_occ_encrypt]
initializiation_data

17


,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs,Start_without_offset,End_without_offset
0,2025/07/22,16:42:45.841870,addClient,1.708µs,AggregationService,0.001708,0.000,0.001708,0.000000,0.001708
1,2025/07/22,16:42:45.854911,DataSpaceClientService - RegisterClient,17.285958ms,Client1,17.285958,13.041,30.326958,0.001708,17.287666
2,2025/07/22,16:42:45.863316,HEService - SetParameters,8.358917ms,Client1,8.358917,21.446,29.804917,17.287666,25.646583
3,2025/07/22,16:42:50.867832,HEService - PartialShareAggregation,1.084µs,Client1,0.001084,5025.962,5025.963084,25.646583,25.647667
4,2025/07/22,16:42:50.872998,EncryptionHandler - handleReceivePublicKey,1.315917ms,Client1,1.315917,5031.128,5032.443917,25.647667,26.963584
5,2025/07/22,16:42:50.904355,HEService - PartialRelinKeyAggregation,2.48075ms,Client1,2.480750,5062.485,5064.965750,26.963584,29.444334
6,2025/07/22,16:42:51.065956,startEncryptionSetupPhaseFor,5.224059s,AggregationService,5224.059000,5224.086,10448.145000,29.444334,5253.503334
7,2025/07/22,16:42:53.846724,addClient,1.292µs,AggregationService,0.001292,8004.854,8004.855292,5253.503334,5253.504626
8,2025/07/22,16:42:53.857312,DataSpaceClientService - RegisterClient,12.761208ms,Client2,12.761208,8015.442,8028.203208,5253.504626,5266.265834
9,2025/07/22,16:42:53.865943,HEService - SetParameters,8.578583ms,Client2,8.578583,8024.073,8032.651583,5266.265834,5274.844417


In [11]:
df = px.data.tips()
fig = px.bar(initializiation_data, base="StartMs", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], labels={"runtime_ms": "Runtime in millisecond", "Service": "Service"}, color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)
fig.show()

In [12]:
request_client_training = data_joined.loc[data_joined["Name"] == "requestClientTraining"].index

print(request_client_training[0])
print(request_client_training[1])

data_after_upload = data_joined[request_client_training[0]:request_client_training[1]]
data_after_upload

17
31


,Date,EndTime,Name,Runtime,Service,runtime_ms,StartMs,EndMs,Start_without_offset,End_without_offset
17,2025/07/22,16:43:09.286822,requestClientTraining,4.1685ms,AggregationService,4.168500,23444.952,23449.120500,10657.377458,10661.545958
18,2025/07/22,16:43:22.080275,HEService - Encrypt,877.91425ms,Client2,877.914250,36238.405,37116.319250,10661.545958,11539.460208
19,2025/07/22,16:43:22.128913,HEService - Encrypt,881.40125ms,Client1,881.401250,36287.043,37168.444250,11539.460208,12420.861458
20,2025/07/22,16:43:23.387515,DataSpaceClientService - UploadData,1.260182542s,Client2,1260.182542,37545.645,38805.827542,12420.861458,13681.044000
21,2025/07/22,16:43:23.424320,DataSpaceClientService - UploadData,1.249576167s,Client1,1249.576167,37582.450,38832.026167,13681.044000,14930.620167
22,2025/07/22,16:43:25.141169,aggregateWeights,56.834709ms,AggregationService,56.834709,39299.299,39356.133709,14930.620167,14987.454876
23,2025/07/22,16:43:25.337108,HEService - PublicKeySwitchGeneration,30.754458ms,Client1,30.754458,39495.238,39525.992458,14987.454876,15018.209334
24,2025/07/22,16:43:25.337140,EncryptionHandler - handlePublicKeySwitch,107.125417ms,Client1,107.125417,39495.270,39602.395417,15018.209334,15125.334751
25,2025/07/22,16:43:25.538703,HEService - PublicKeySwitchGeneration,30.010208ms,Client2,30.010208,39696.833,39726.843208,15125.334751,15155.344959
26,2025/07/22,16:43:25.538735,EncryptionHandler - handlePublicKeySwitch,106.422542ms,Client2,106.422542,39696.865,39803.287542,15155.344959,15261.767501


In [20]:
df = px.data.tips()
fig = px.bar(data_after_upload, base="StartMs", x="runtime_ms", y="Service", color='Name', orientation='h',
             hover_data=["EndTime", "StartMs", "runtime_ms"], labels={"runtime_ms": "Runtime in millisecond", "Service": "Service"}, color_discrete_sequence=px.colors.qualitative.Safe,
             height=400)

max_box = data_after_upload.iloc[1]["StartMs"]-300
min_box = data_after_upload.iloc[0]["EndMs"]+300
fig.add_shape(
    type="rect",
    x0=min_box,  # Start x position
    y0=-0.4,  # Start y position (below the bars)
    x1=max_box,   # End x position
    y1=2.4,  # End y position
    fillcolor="lightblue",  # Pastel blue color
    opacity=0.6,
    line=dict(color="lightblue", width=1),
)
fig.add_annotation(
    x=min_box + (max_box-min_box)/2,  # Center of the rectangle
    y=(2.4-0.4)/2+0.1,  # Position above the rectangle
    text="Training Phase of the two python ml clients",
    showarrow=False,
    font=dict(size=12, color="black"),
    opacity=0.8
)

fig.show()